# Nigeria Inflation Forecast

Nigerian households and small businesses set prices, salaries and savings without knowing where inflation is heading.
Headline inflation peaked above 30% in 2024, the CPI was rebased in January 2025, and inflation has since fallen to about 14.6% (September 2026).
**Will that decline continue, level off, or reverse?**

In this notebook we pull Nigeria's monthly inflation, crude oil prices and the naira/dollar exchange rate from the Central Bank of Nigeria (CBN),
explore them, compare forecasting models against a naive baseline, and forecast the next 6–12 months with an uncertainty range.

## 1. Setup

We load the libraries we need:

- `pandas`: tables (DataFrames) and time-series tools
- `requests`: downloads data from a web address (URL)
- `Path`: builds file paths that work on Windows, Mac and Linux

In [ ]:
from pathlib import Path

import pandas as pd
import requests

## 2. Data pull

**Question:** Can we pull Nigeria's monthly inflation, crude oil price and naira/dollar rate straight from the CBN and combine them into one monthly table?

The data pages on cbn.gov.ng load their numbers from hidden **JSON feeds**. JSON is a text format for data: here, each feed is a long list of
records, and each record (one month or one day) becomes a row in our table.

These feeds are **undocumented**, so the CBN could change or remove them without warning. That's why we always save a CSV copy after a
successful pull, and fall back to that copy if the download fails.

In [ ]:
# The three hidden JSON feeds behind cbn.gov.ng data pages (undocumented, so they may change)
BASE_URL = "https://www.cbn.gov.ng/api/"
INFLATION_ENDPOINT = "GetAllInflationRates"
CRUDE_ENDPOINT = "GetAllDailyCrude"
FX_ENDPOINT = "GetAllExchangeRates"

# Some servers refuse requests that don't look like they come from a web browser,
# so we send the same "User-Agent" label a normal browser would
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/130.0 Safari/537.36"
}

# Where we keep our saved copy of the cleaned monthly table
CSV_PATH = Path("data") / "nigeria_macro.csv"

### 2.1 Download the three feeds

`requests.get()` asks the server for a URL and returns a **response**. Every response has a **status code**: `200` means OK,
`403` means "forbidden" (blocked), `404` means "not found". `raise_for_status()` turns any bad status code into an error, so a
failed download can't slip through silently. `timeout=60` means: give up after 60 seconds instead of waiting forever.

We wrap the downloads in `try` / `except`. If **anything** goes wrong (no internet, the server blocks us, the reply isn't valid JSON),
we print the problem and set `live_data = False`. Later cells then load the saved CSV instead of crashing.

In [ ]:
def fetch_json(endpoint):
    # Download one CBN feed and return it as a list of records (dictionaries)
    url = BASE_URL + endpoint
    response = requests.get(url, headers=HEADERS, timeout=60)
    response.raise_for_status()  # stop with an error if the server says no (e.g. 403 or 404)
    records = response.json()
    print(f"{endpoint}: {len(records):,} records")
    return records

In [ ]:
try:
    inflation_raw = fetch_json(INFLATION_ENDPOINT)
    crude_raw = fetch_json(CRUDE_ENDPOINT)
    fx_raw = fetch_json(FX_ENDPOINT)
    live_data = True
except (requests.RequestException, ValueError) as error:
    # RequestException = network problem or bad status code; ValueError = the reply wasn't valid JSON
    print("Could not get live data from the CBN API:", error)
    print("We'll load the saved CSV copy instead.")
    live_data = False

Let's look at one raw record from each feed before cleaning anything.

*Cells that start with `if live_data:` only do something when the download worked.*

In [ ]:
if live_data:
    print(inflation_raw[0])
    print(crude_raw[0])
    print(fx_raw[0])

**Finding:** All three feeds downloaded (285 monthly inflation records, about 4,000 daily crude prices and about 62,000 daily exchange rates).
Three things to fix while cleaning:
1. **Every number is text.** Notice the quotes in `'allItemsYearOn': '14.63'`. We can't do maths on text, so we convert with `pd.to_numeric`.
2. **Newest first.** Time series should run oldest → newest.
3. **Different date styles.** Inflation has separate year and month numbers, crude uses `dd/mm/yyyy`, and exchange rates use `yyyy-mm-dd`.

### 2.2 Clean the inflation feed

We keep three **year-on-year** rates. "Year-on-year" means: how much higher prices are than in the same month last year.

| Feed column | Our name | Meaning |
|---|---|---|
| `allItemsYearOn` | `Headline` | All goods and services |
| `foodYearOn` | `Food` | Food only |
| `allItemsLessFrmProdAndEnergyYearOn` | `Core` | Everything except farm produce and energy, the less volatile part |

We build a proper date from `tyear` and `tmonth`, using **day 1** of each month. Every series in this project will use
month-start dates, so they line up exactly when we combine them.

In [ ]:
def clean_inflation(records):
    # Turn the raw inflation records into a monthly table, oldest month first
    df = pd.DataFrame(records)

    # Build a real date from the year and month columns (day 1 = "month start")
    df["Date"] = pd.to_datetime({"year": df["tyear"], "month": df["tmonth"], "day": 1})

    # Keep only the year-on-year rates we need, with short names
    df = df.rename(columns={
        "allItemsYearOn": "Headline",
        "foodYearOn": "Food",
        "allItemsLessFrmProdAndEnergyYearOn": "Core",
    })
    df = df[["Date", "Headline", "Food", "Core"]]

    # Convert text like "14.63" to numbers; anything that isn't a number becomes NaN (blank),
    # which our checks in section 2.6 will catch
    for column in ["Headline", "Food", "Core"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    # The feed is newest-first; time series should run oldest-first
    df = df.set_index("Date").sort_index()
    return df

In [ ]:
if live_data:
    inflation = clean_inflation(inflation_raw)
    print(inflation.dtypes)
    display(inflation.tail())

**Finding:** The inflation table has 285 months, from January 2003 to September 2026, with no gaps. The three columns are now
`float64` (decimal numbers) instead of text. In September 2026, headline inflation was **14.63%**, food **19.59%** and core **12.95%**.

### 2.3 Clean the crude oil feed

This is the daily price of **Bonny Light**, Nigeria's main export crude, in US dollars per barrel. Two techniques are new here:

- **An explicit date format.** A date like `03/04/2020` could be 3 April or 4 March. We tell pandas the exact format (`%d/%m/%Y`
  = day/month/year) so it can't guess wrong.
- **Resampling.** Inflation is monthly but crude is daily. `resample("MS").mean()` groups all the days in each month and takes the
  average. `"MS"` means "label each group by its **M**onth **S**tart", so January 2020's average gets the date 2020-01-01, the same
  as January 2020's inflation.

In [ ]:
def clean_crude(records):
    # Turn daily crude oil prices into monthly average prices (US$ per barrel)
    df = pd.DataFrame(records)

    # Dates look like "22/10/2026" (day/month/year), so give pandas the exact format
    df["Date"] = pd.to_datetime(df["postDate"], format="%d/%m/%Y")
    df["Crude_USD"] = pd.to_numeric(df["crudeOilPrice"], errors="coerce")

    # Average all the days in each month (this also absorbs the few duplicate days in the feed)
    monthly = df.set_index("Date")["Crude_USD"].resample("MS").mean()
    return monthly.round(2)

In [ ]:
if live_data:
    crude_monthly = clean_crude(crude_raw)
    print("First month:", crude_monthly.index.min().date(), "| Last month:", crude_monthly.index.max().date())

    # resample() creates a row for every month, so a month with no daily prices shows up as NaN
    empty_months = crude_monthly[crude_monthly.isna()].index.strftime("%Y-%m").tolist()
    print("Months with no crude prices:", empty_months)
    display(crude_monthly["2020-01":"2020-06"])

**Finding:** Crude prices start in **October 2009**, so the combined table will have no crude price for 2003 to September 2009.
**June and July 2019 are missing** from the CBN feed entirely. We'll decide how to handle these gaps in Phase 3.
The April 2020 average is very low, with a few days at $7–10 a barrel. That's not an error: it's the COVID-19 oil price
crash, when lockdowns cut global oil demand.

### 2.4 Clean the exchange-rate feed

This feed has daily official rates for **every** currency, and we only want the US dollar. CLAUDE.md warns that the currency names are messy,
so let's **look before we filter**. `repr()` prints each name with quote marks around it, which makes hidden spaces and tabs (`\t`) visible.

In [ ]:
if live_data:
    fx = pd.DataFrame(fx_raw)
    # repr() shows hidden characters, e.g. 'US DOLLAR ' with a trailing space
    for name in sorted(fx["currency"].unique()):
        print(repr(name))

**Finding:** The names are messy. There's `'US DOLLAR'` *and* `'US DOLLAR '` (trailing space), `'EURO\t'` (a hidden tab),
and spelling variants like `'DANISH KRONA'` / `'DANISH KRONER'` and `'POUND STERLING'` / `'POUNDS STERLING'`.
If we filtered for `"US DOLLAR"` without cleaning, we'd silently lose the rows with a space. `.str.strip()` removes spaces and tabs
from both ends of each name, which is all the US dollar needs.

Each day has three rates: **buying**, **selling**, and **central** (the official midpoint between them). We use the **central rate**,
in naira per US dollar, then average each month just like crude.

In [ ]:
def clean_usd_ngn(records):
    # Pick out the US dollar rows and turn daily rates into monthly averages (naira per dollar)
    df = pd.DataFrame(records)

    # Remove hidden spaces/tabs so "US DOLLAR " and "US DOLLAR" count as the same currency
    df["currency"] = df["currency"].str.strip()
    usd = df[df["currency"] == "US DOLLAR"].copy()

    usd["Date"] = pd.to_datetime(usd["ratedate"])  # already in yyyy-mm-dd format
    usd["USD_NGN"] = pd.to_numeric(usd["centralrate"], errors="coerce")

    monthly = usd.set_index("Date")["USD_NGN"].resample("MS").mean()
    return monthly.round(2)

In [ ]:
if live_data:
    usd_ngn_monthly = clean_usd_ngn(fx_raw)
    print("First month:", usd_ngn_monthly.index.min().date(), "| Last month:", usd_ngn_monthly.index.max().date())
    print("Months with no rate:", usd_ngn_monthly.isna().sum())
    # Around the June 2023 float of the naira
    display(usd_ngn_monthly["2023-03":"2023-09"])

**Finding:** The US dollar rate runs from December 2001 with no missing months. The June 2023 float is easy to see: about
**₦461** per dollar in May 2023, **₦608** in June and **₦770** in July. This is a jump of roughly two-thirds in two months. Jumps like this
are **policy decisions**, not random noise, and we'll keep that in mind when modelling.

### 2.5 Combine into one monthly table, save it, or fall back to the saved copy

We use the inflation table as the "spine" and **join** the crude and FX columns onto it, matching rows by `Date`.
`join` keeps only the months in the inflation table, so months the other feeds have but inflation doesn't
(e.g. October 2026, which is only partly over) drop out.

If the download worked, we save the result to `data/nigeria_macro.csv`. If it didn't, we load the last saved copy.

In [ ]:
if live_data:
    inflation = clean_inflation(inflation_raw)
    crude_monthly = clean_crude(crude_raw)
    usd_ngn_monthly = clean_usd_ngn(fx_raw)

    macro = inflation.join(crude_monthly).join(usd_ngn_monthly)

    CSV_PATH.parent.mkdir(exist_ok=True)  # make sure the data/ folder exists
    macro.to_csv(CSV_PATH)
    print("Live data pulled and saved to", CSV_PATH)
else:
    macro = pd.read_csv(CSV_PATH, index_col="Date", parse_dates=True)
    print("Loaded saved copy from", CSV_PATH)

### 2.6 Check the combined table

After every merge we check four things: the **date range**, **missing months**, **duplicate months** and **blank values**.
A merge can silently create any of these problems.

In [ ]:
print("Rows, columns:", macro.shape)
print("From", macro.index.min().date(), "to", macro.index.max().date())

# Every month between the first and last date should appear exactly once
all_months = pd.date_range(macro.index.min(), macro.index.max(), freq="MS")
print("Missing months:", len(all_months.difference(macro.index)))
print("Duplicate months:", macro.index.duplicated().sum())

# Count blank (NaN) values in each column
print("\nBlank values per column:")
print(macro.isna().sum())

In [ ]:
display(macro.head())
display(macro.tail())

**Finding:** We now have one clean monthly table: **285 months** (January 2003 to September 2026) with columns `Headline`, `Food`, `Core`,
`Crude_USD` and `USD_NGN`, saved to `data/nigeria_macro.csv`. There are no missing or duplicate months. The only blanks are in
`Crude_USD`: 81 months before the crude feed starts (January 2003 to September 2009) plus the June and July 2019 gap, so **83** in total.

Still to come in Phase 3 (data quality): add the `Rebased` marker for the January 2025 CPI rebasing, decide what to do with the crude gaps,
and spot-check the CBN numbers against NBS press releases.